In [ ]:
import pandas as pd
import re

In [ ]:
COLUMNS = ["sentiment", "id", "date", "query", "user", "text"]

def clean_binary(input_path: str, output_path: str, sample_size: int = None):
    df = pd.read_csv(
        input_path,
        encoding="latin-1",
        header=None,
        names=COLUMNS
    )

    # Keep only sentiment + text
    df = df[["sentiment", "text"]]

    # Map labels
    # 0 = Negative, 4 = Positive  -->  0 = Negative, 1 = Positive
    df["sentiment"] = df["sentiment"].map({0: 0, 4: 1})

    # Clean text (emoji-aware)
    df["text"] = df["text"].apply(clean_text)

    # Drop empty rows
    df = df[df["text"].str.len() > 0]

    # Optional sampling 
    if sample_size:
        df = df.sample(sample_size, random_state=42)

    df.to_csv(output_path, index=False)
    print(f" Binary csv cleaned: {len(df)} rows saved to {output_path}")

In [ ]:
# Simple emoji-to-token mapping 
EMOJI_MAP = {
    "❤️": "emoji_love",
    "❤": "emoji_love",
    "😍": "emoji_love",
    "😊": "emoji_happy",
    "😁": "emoji_happy",
    "😂": "emoji_laugh",
    "🤣": "emoji_laugh",
    "😭": "emoji_cry",
    "😢": "emoji_sad",
    "😡": "emoji_angry",
    "🔥": "emoji_fire",
    "👏": "emoji_clap",
    "😮": "emoji_wow",
    "👍": "emoji_thumbsup",
    "👎": "emoji_thumbsdown",
    "💯": "emoji_100",
}


def replace_emojis(text: str) -> str:
    for emoji, token in EMOJI_MAP.items():
        text = text.replace(emoji, f" {token} ")
    return text


def handle_negation(text: str) -> str:
    """
    Converts:
      'not good'  -> 'not_good'
      'never liked' -> 'never_liked'
      'no way' -> 'no_way'

    This helps the model learn sentiment flips properly.
    """
    text = re.sub(r"\b(not|never|no)\s+([a-z]+)\b", r"\1_\2", text)
    return text

def clean_text(text: str) -> str:
    text = str(text).lower()

    # Convert emojis into meaningful tokens instead of deleting them
    text = replace_emojis(text)

    # Remove URLs, mentions
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)

    # Keep hashtags text but remove # symbol
    text = re.sub(r"#", "", text)

    # Negation handling (before removing characters)
    text = handle_negation(text)

    # Keep only letters, spaces, and underscore (for emoji tokens)
    text = re.sub(r"[^a-z_\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
# Paths 
BINARY_RAW = "data/raw/binary.csv"
BINARY_OUT = "data/cleaned/binary_cleaned.csv"

# Clean Binary Dataset
clean_binary(BINARY_RAW, BINARY_OUT, sample_size=None)